# 🏠 House Price Prediction — Pipeline ML Complet sur Snowflake

**Objectif** : Construire un pipeline end-to-end de Data Engineering et Machine Learning directement dans Snowflake, sans exporter les données.

| Étape | Description |
|-------|-------------|
| 1 | Configuration de la session Snowpark |
| 2 | Ingestion & exploration des données |
| 3 | Analyse exploratoire (EDA) |
| 4 | Feature Engineering |
| 5 | Entraînement des modèles |
| 6 | Évaluation des performances |
| 7 | Optimisation des hyperparamètres |
| 8 | Model Registry |
| 9 | Inférence |


---
## ⚙️ ÉTAPE 1 — Configuration de la Session Snowpark

> **Snowpark** est l'API Python de Snowflake. Elle permet de manipuler les données directement dans Snowflake avec une syntaxe proche de Pandas/PySpark, sans déplacer les données.

In [ ]:
# -------------------------------------------------------------------
# Import des bibliothèques
# -------------------------------------------------------------------
import warnings
warnings.filterwarnings('ignore')

# Snowpark & Snowflake ML
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import IntegerType, FloatType, StringType

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Machine Learning
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json
import os

print('✅ Toutes les bibliothèques sont importées avec succès')

In [ ]:
# -------------------------------------------------------------------
# Connexion à la session Snowpark active (dans un Snowflake Notebook,
# la session est automatiquement disponible via get_active_session())
# -------------------------------------------------------------------
session = get_active_session()

# Configuration du contexte
session.use_database('HOUSE_PRICE_DB')
session.use_schema('RAW')
session.use_warehouse('HOUSE_PRICE_WH')
session.use_role('HOUSE_PRICE_ROLE')

# Vérification
print(f'✅ Connecté à Snowflake')
print(f'   Database  : {session.get_current_database()}')
print(f'   Schema    : {session.get_current_schema()}')
print(f'   Warehouse : {session.get_current_warehouse()}')
print(f'   Role      : {session.get_current_role()}')

---
## 📥 ÉTAPE 2 — Ingestion & Exploration initiale des données

> On charge les données depuis la table Snowflake dans un **Snowpark DataFrame** (les données restent dans Snowflake, seules les métadonnées sont rapatriées).

In [ ]:
# -------------------------------------------------------------------
# Chargement du dataset depuis Snowflake
# -------------------------------------------------------------------
df_snow = session.table('HOUSE_PRICE_DB.RAW.HOUSE_PRICES')

print(f'✅ Dataset chargé : {df_snow.count()} lignes')
print(f'   Colonnes : {len(df_snow.columns)}')
print()

# Affichage du schéma
print('📋 Schéma du dataset :')
df_snow.printSchema()

In [ ]:
# -------------------------------------------------------------------
# Aperçu des premières lignes
# -------------------------------------------------------------------
print('🔍 Aperçu des 5 premières lignes :')
df_snow.show(5)

# Conversion en Pandas pour l'analyse exploratoire
# (On ramène les données en local une seule fois pour l'EDA)
df = df_snow.drop('_INGESTED_AT').to_pandas()
df.columns = [c.lower() for c in df.columns]

print(f'\n📊 Shape du DataFrame Pandas : {df.shape}')

In [ ]:
# -------------------------------------------------------------------
# Statistiques descriptives de base
# -------------------------------------------------------------------
print('📈 Statistiques descriptives :')
display(df.describe().round(2))

print('\n🔎 Types de colonnes :')
print(df.dtypes)

print('\n❓ Valeurs manquantes par colonne :')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '   Aucune valeur manquante ✅')

---
## 🔍 ÉTAPE 3 — Analyse Exploratoire des Données (EDA)

> L'EDA permet de **comprendre les distributions**, détecter les outliers et analyser les **corrélations** entre variables.

In [ ]:
# -------------------------------------------------------------------
# 3.1 Distribution de la variable cible : PRICE
# -------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution du Prix de Vente', fontsize=14, fontweight='bold')

# Histogramme
axes[0].hist(df['price'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution brute')
axes[0].set_xlabel('Prix (₹)')
axes[0].set_ylabel('Fréquence')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', label=f'Moyenne : {df["price"].mean():,.0f}')
axes[0].axvline(df['price'].median(), color='orange', linestyle='--', label=f'Médiane : {df["price"].median():,.0f}')
axes[0].legend()

# Log-transform
axes[1].hist(np.log1p(df['price']), bins=40, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution log-transformée')
axes[1].set_xlabel('log(Prix)')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.savefig('/tmp/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Statistiques du prix :')
print(f'   Min    : {df["price"].min():>12,.0f}')
print(f'   Max    : {df["price"].max():>12,.0f}')
print(f'   Moyenne: {df["price"].mean():>12,.0f}')
print(f'   Médiane: {df["price"].median():>12,.0f}')
print(f'   Écart-type: {df["price"].std():>9,.0f}')

In [ ]:
# -------------------------------------------------------------------
# 3.2 Distribution des variables numériques
# -------------------------------------------------------------------
num_cols = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution des Variables Numériques', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col.capitalize()}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Fréquence')

# Boxplot du prix par nombre de chambres
df.boxplot(column='price', by='bedrooms', ax=axes[5], 
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red'))
axes[5].set_title('Prix par nombre de chambres')
axes[5].set_xlabel('Chambres')
axes[5].set_ylabel('Prix')
plt.suptitle('')

plt.tight_layout()
plt.savefig('/tmp/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -------------------------------------------------------------------
# 3.3 Impact des variables catégorielles sur le prix
# -------------------------------------------------------------------
cat_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
            'airconditioning', 'prefarea', 'furnishingstatus']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('Prix Moyen selon les Variables Catégorielles', fontsize=14, fontweight='bold')
axes = axes.flatten()

colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800', '#00BCD4', '#E91E63']

for i, col in enumerate(cat_cols):
    avg_price = df.groupby(col)['price'].mean().sort_values(ascending=False)
    bars = axes[i].bar(avg_price.index, avg_price.values, 
                       color=colors[i % len(colors)], alpha=0.85, edgecolor='white')
    axes[i].set_title(f'{col}')
    axes[i].set_ylabel('Prix moyen')
    axes[i].tick_params(axis='x', rotation=15)
    # Annotations
    for bar, val in zip(bars, avg_price.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                    f'{val:,.0f}', ha='center', va='bottom', fontsize=7)

axes[7].axis('off')
plt.tight_layout()
plt.savefig('/tmp/categorical_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -------------------------------------------------------------------
# 3.4 Matrice de corrélation
# -------------------------------------------------------------------

# Encodage temporaire des variables catégorielles pour la corrélation
df_encoded_corr = df.copy()
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df_encoded_corr[col] = (df_encoded_corr[col] == 'yes').astype(int)

furnishing_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df_encoded_corr['furnishingstatus'] = df_encoded_corr['furnishingstatus'].map(furnishing_map)

# Matrice de corrélation
corr_matrix = df_encoded_corr.corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    center=0,
    mask=mask,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Matrice de Corrélation — House Price Dataset', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/tmp/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Corrélations avec le prix
print('\n🎯 Corrélations avec le PRIX (triées) :')
price_corr = corr_matrix['price'].drop('price').sort_values(ascending=False)
for col, val in price_corr.items():
    bar = '█' * int(abs(val) * 20)
    sign = '+' if val > 0 else '-'
    print(f'   {col:<20} {sign}{abs(val):.3f}  {bar}')

---
## 🔧 ÉTAPE 4 — Feature Engineering & Préparation des Données

> On transforme les données brutes en **features exploitables** par les modèles ML :
> - Encodage des variables catégorielles
> - Normalisation des variables numériques
> - Séparation train / test

In [ ]:
# -------------------------------------------------------------------
# 4.1 Copie et encodage des features
# -------------------------------------------------------------------
df_ml = df.copy()

# Encodage binaire (yes=1 / no=0)
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
               'airconditioning', 'prefarea']
for col in binary_cols:
    df_ml[col] = (df_ml[col] == 'yes').astype(int)
    print(f'   ✅ {col}: encodé (yes=1 / no=0)')

# Encodage ordinal du statut d'ameublement
furnishing_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df_ml['furnishingstatus'] = df_ml['furnishingstatus'].map(furnishing_map)
print(f'   ✅ furnishingstatus: encodé (furnished=2 / semi-furnished=1 / unfurnished=0)')

# Feature engineering : création de nouvelles features
df_ml['price_per_sqft'] = df_ml['price'] / df_ml['area']      # prix au m²
df_ml['room_ratio']     = df_ml['bathrooms'] / df_ml['bedrooms']  # ratio sdb/chambres
df_ml['total_amenities'] = (df_ml['guestroom'] + df_ml['basement'] +
                             df_ml['hotwaterheating'] + df_ml['airconditioning'])
print('\n   ✅ Nouvelles features créées :')
print('      - price_per_sqft   (prix au m²)')
print('      - room_ratio       (ratio sdb / chambres)')
print('      - total_amenities  (nb équipements premium)')

print(f'\n📊 Dataset après feature engineering : {df_ml.shape}')
display(df_ml.head(3))

In [ ]:
# -------------------------------------------------------------------
# 4.2 Séparation X / y et Train / Test
# -------------------------------------------------------------------

# Features (on retire les colonnes dérivées du prix pour éviter la fuite de données)
feature_cols = [
    'area', 'bedrooms', 'bathrooms', 'stories',
    'mainroad', 'guestroom', 'basement', 'hotwaterheating',
    'airconditioning', 'parking', 'prefarea', 'furnishingstatus',
    'room_ratio', 'total_amenities'
]

X = df_ml[feature_cols]
y = df_ml['price']

# Split 80% train / 20% test avec seed fixe pour reproductibilité
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'✅ Séparation Train / Test effectuée :')
print(f'   Train : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'   Test  : {X_test.shape[0]} lignes  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'   Features : {len(feature_cols)} colonnes')

# Normalisation des variables numériques
num_features = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'room_ratio', 'total_amenities']
cat_features = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
                'airconditioning', 'prefarea', 'furnishingstatus']

preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), num_features),
    ('passthrough', 'passthrough', cat_features)
])

print('\n✅ Preprocessor créé (StandardScaler sur features numériques)')

In [ ]:
# -------------------------------------------------------------------
# 4.3 Sauvegarde des datasets transformés dans Snowflake
# -------------------------------------------------------------------

# Ajout du split label
df_train = pd.concat([X_train, y_train], axis=1)
df_train['SPLIT'] = 'TRAIN'
df_test = pd.concat([X_test, y_test], axis=1)
df_test['SPLIT'] = 'TEST'
df_features_full = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
df_features_full.columns = [c.upper() for c in df_features_full.columns]

# Écriture dans Snowflake
snow_features = session.create_dataframe(df_features_full)
snow_features.write.save_as_table(
    'HOUSE_PRICE_DB.ANALYTICS.FEATURES_ML',
    mode='overwrite'
)

print('✅ Table HOUSE_PRICE_DB.ANALYTICS.FEATURES_ML sauvegardée')
print(f'   Lignes : {df_features_full.shape[0]}')
print(f'   Colonnes : {df_features_full.shape[1]}')

---
## 🤖 ÉTAPE 5 — Entraînement des Modèles ML

> On entraîne **3 modèles de régression** pour comparer leurs performances :
> - **Linear Regression** : modèle de référence (baseline)
> - **Ridge Regression** : régression linéaire avec régularisation L2
> - **Random Forest** : ensemble de 100 arbres de décision


In [ ]:
# -------------------------------------------------------------------
# 5.1 Définition des modèles et entraînement
# -------------------------------------------------------------------

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Entraîne un modèle et retourne ses métriques."""
    model.fit(X_tr, y_tr)
    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)
    
    # Métriques sur le test set
    mae   = mean_absolute_error(y_te, y_pred_test)
    rmse  = np.sqrt(mean_squared_error(y_te, y_pred_test))
    r2    = r2_score(y_te, y_pred_test)
    r2_tr = r2_score(y_tr, y_pred_train)
    
    # Cross-validation (5 folds)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    
    return {
        'Modèle'   : name,
        'MAE'      : mae,
        'RMSE'     : rmse,
        'R² Test'  : r2,
        'R² Train' : r2_tr,
        'CV R² Moy': cv_scores.mean(),
        'CV R² Std': cv_scores.std(),
        'model_obj': model,
        'y_pred'   : y_pred_test
    }


# Pipelines avec preprocessing
models = {
    'Linear Regression': Pipeline([
        ('prep', preprocessor),
        ('model', LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('prep', preprocessor),
        ('model', Ridge(alpha=10.0))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ])
}

# Entraînement
results = []
trained_models = {}

for name, pipeline in models.items():
    print(f'⏳ Entraînement : {name}...')
    res = evaluate_model(name, pipeline, X_train, y_train, X_test, y_test)
    trained_models[name] = res['model_obj']
    results.append({k: v for k, v in res.items() if k not in ['model_obj', 'y_pred']})
    print(f'   ✅ R²={res["R² Test"]:.4f} | MAE={res["MAE"]:,.0f} | RMSE={res["RMSE"]:,.0f}')

df_results = pd.DataFrame(results)
print('\n📊 Tableau comparatif des modèles :')
display(df_results.set_index('Modèle').round(4))

---
## 📊 ÉTAPE 6 — Évaluation des Performances

> On visualise les performances de chaque modèle et on analyse les résidus.

In [ ]:
# -------------------------------------------------------------------
# 6.1 Visualisation comparative des métriques
# -------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparaison des Performances — 3 Modèles', fontsize=14, fontweight='bold')

model_names = df_results['Modèle'].tolist()
colors = ['#2196F3', '#FF9800', '#4CAF50']

for ax, metric, title in zip(axes, ['R² Test', 'MAE', 'RMSE'],
                              ['R² (plus haut = meilleur)', 'MAE (plus bas = meilleur)', 'RMSE (plus bas = meilleur)']):
    bars = ax.bar(model_names, df_results[metric], color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, df_results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.4f}' if metric == 'R² Test' else f'{val:,.0f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -------------------------------------------------------------------
# 6.2 Graphiques Prédictions vs Réelles pour chaque modèle
# -------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Prédictions vs Valeurs Réelles', fontsize=14, fontweight='bold')

for ax, (name, model), color in zip(axes, trained_models.items(), colors):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    
    ax.scatter(y_test, y_pred, alpha=0.6, color=color, s=30)
    # Ligne parfaite
    min_val, max_val = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Parfait')
    ax.set_title(f'{name}\nR² = {r2:.4f}')
    ax.set_xlabel('Prix Réel')
    ax.set_ylabel('Prix Prédit')
    ax.legend()

plt.tight_layout()
plt.savefig('/tmp/predictions_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ⚡ ÉTAPE 7 — Optimisation des Hyperparamètres

> On optimise le **Random Forest** (meilleur modèle de base) avec **RandomizedSearchCV** pour trouver la combinaison d'hyperparamètres optimale.

In [ ]:
# -------------------------------------------------------------------
# 7.1 RandomizedSearchCV sur Random Forest
# -------------------------------------------------------------------
from scipy.stats import randint, uniform

print('⏳ Optimisation des hyperparamètres (RandomizedSearchCV)...')
print('   Cela peut prendre 1-2 minutes...\n')

# Espace de recherche
param_dist = {
    'model__n_estimators'     : randint(50, 300),
    'model__max_depth'        : [None, 5, 10, 15, 20],
    'model__min_samples_split': randint(2, 10),
    'model__min_samples_leaf' : randint(1, 5),
    'model__max_features'     : ['sqrt', 'log2', None]
}

rf_base = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f'\n✅ Meilleurs hyperparamètres trouvés :')
for param, val in random_search.best_params_.items():
    print(f'   {param.replace("model__", ""):<25} : {val}')
print(f'\n   Meilleur R² CV : {random_search.best_score_:.4f}')

In [ ]:
# -------------------------------------------------------------------
# 7.2 Évaluation du meilleur modèle optimisé
# -------------------------------------------------------------------
best_model = random_search.best_estimator_
y_pred_best = best_model.predict(X_test)

# Métriques finales
mae_best  = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best   = r2_score(y_test, y_pred_best)
r2_train_best = r2_score(y_train, best_model.predict(X_train))

# Métriques du Random Forest de base pour comparaison
rf_base_model = trained_models['Random Forest']
y_pred_rf_base = rf_base_model.predict(X_test)
r2_rf_base = r2_score(y_test, y_pred_rf_base)
mae_rf_base = mean_absolute_error(y_test, y_pred_rf_base)

print('\n📊 Comparaison avant / après optimisation :')
print(f'   {"Métrique":<12} {"RF Base":>12} {"RF Optimisé":>14} {"Amélioration":>14}')
print('   ' + '-' * 55)
print(f'   {"R² Test":<12} {r2_rf_base:>12.4f} {r2_best:>14.4f} {(r2_best-r2_rf_base)*100:>+13.2f}%')
print(f'   {"MAE":<12} {mae_rf_base:>12,.0f} {mae_best:>14,.0f} {(mae_rf_base-mae_best)/mae_rf_base*100:>+13.2f}%')
print(f'   {"RMSE":<12} {np.sqrt(mean_squared_error(y_test, y_pred_rf_base)):>12,.0f} {rmse_best:>14,.0f}')

# Sauvegarde des métriques finales
final_metrics = {
    'model_name' : 'RandomForest_Optimized',
    'mae'        : float(mae_best),
    'rmse'       : float(rmse_best),
    'r2_test'    : float(r2_best),
    'r2_train'   : float(r2_train_best),
    'best_params': {k.replace('model__',''):str(v) for k,v in random_search.best_params_.items()}
}

print(f'\n🏆 Meilleur modèle : R² = {r2_best:.4f} | MAE = {mae_best:,.0f}')

In [ ]:
# -------------------------------------------------------------------
# 7.3 Importance des features
# -------------------------------------------------------------------
rf_model_obj = best_model.named_steps['model']
feature_names_out = num_features + cat_features
importances = pd.Series(rf_model_obj.feature_importances_, index=feature_names_out)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors_imp = ['#4CAF50' if v > importances.mean() else '#90A4AE' for v in importances]
bars = plt.barh(importances.index, importances.values, color=colors_imp, edgecolor='white')
plt.axvline(importances.mean(), color='red', linestyle='--', alpha=0.7, label='Moyenne')
plt.xlabel('Importance')
plt.title('Importance des Features — Random Forest Optimisé', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('/tmp/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📌 Top 5 features les plus importantes :')
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f'   {feat:<22} : {imp:.4f} ({imp*100:.1f}%)')

---
## 🗄️ ÉTAPE 8 — Stockage dans le Snowflake Model Registry

> Le **Model Registry** de Snowflake permet de versionner, documenter et déployer des modèles ML directement dans la plateforme, avec gouvernance et traçabilité.

In [ ]:
# -------------------------------------------------------------------
# 8.1 Enregistrement du modèle dans le Snowflake Model Registry
# -------------------------------------------------------------------
from snowflake.ml.registry import Registry
import snowflake.ml.model as mlmodel

# Initialisation du registry
registry = Registry(
    session=session,
    database_name='HOUSE_PRICE_DB',
    schema_name='ML'
)

print('✅ Model Registry initialisé : HOUSE_PRICE_DB.ML')

# Création d'un sample d'input pour le schéma du modèle
X_sample = X_test.head(10)

# Log du modèle dans le registry
model_ref = registry.log_model(
    model=best_model,
    model_name='HOUSE_PRICE_RF_MODEL',
    version_name='V1',
    comment='Random Forest optimisé avec RandomizedSearchCV - Pipeline complet avec preprocessing',
    metrics={
        'mae'     : final_metrics['mae'],
        'rmse'    : final_metrics['rmse'],
        'r2_test' : final_metrics['r2_test'],
        'r2_train': final_metrics['r2_train']
    },
    sample_input_data=X_sample,
    conda_dependencies=['scikit-learn', 'pandas', 'numpy']
)

print(f'\n✅ Modèle enregistré avec succès !')
print(f'   Nom     : HOUSE_PRICE_RF_MODEL')
print(f'   Version : V1')
print(f'   R² Test : {final_metrics["r2_test"]:.4f}')
print(f'   MAE     : {final_metrics["mae"]:,.0f}')

In [ ]:
# -------------------------------------------------------------------
# 8.2 Listage des modèles dans le registry
# -------------------------------------------------------------------
print('📋 Modèles disponibles dans le registry :')
models_list = registry.show_models()
display(models_list)

print('\n📋 Versions du modèle HOUSE_PRICE_RF_MODEL :')
model_versions = registry.get_model('HOUSE_PRICE_RF_MODEL').show_versions()
display(model_versions)

---
## 🔮 ÉTAPE 9 — Inférence avec le Modèle du Registry

> On charge le modèle depuis le registry et on l'utilise pour faire des prédictions sur de **nouvelles données** simulées.

In [ ]:
# -------------------------------------------------------------------
# 9.1 Chargement du modèle depuis le registry
# -------------------------------------------------------------------
loaded_model = registry.get_model('HOUSE_PRICE_RF_MODEL').version('V1')
print('✅ Modèle chargé depuis le registry : HOUSE_PRICE_RF_MODEL V1')

# -------------------------------------------------------------------
# 9.2 Simulation de nouvelles données à prédire
# -------------------------------------------------------------------
new_houses = pd.DataFrame({
    'area'             : [3000, 5500, 1800, 7000, 2500],
    'bedrooms'         : [3,    5,    2,    6,    3   ],
    'bathrooms'        : [2,    3,    1,    4,    2   ],
    'stories'          : [2,    3,    1,    4,    2   ],
    'mainroad'         : [1,    1,    0,    1,    1   ],
    'guestroom'        : [0,    1,    0,    1,    0   ],
    'basement'         : [1,    1,    0,    1,    0   ],
    'hotwaterheating'  : [0,    1,    0,    1,    0   ],
    'airconditioning'  : [1,    1,    0,    1,    1   ],
    'parking'          : [1,    2,    0,    3,    1   ],
    'prefarea'         : [0,    1,    0,    1,    0   ],
    'furnishingstatus' : [2,    2,    0,    2,    1   ],
    'room_ratio'       : [0.67, 0.6,  0.5,  0.67, 0.67],
    'total_amenities'  : [2,    4,    0,    4,    1   ]
})

house_labels = ['Maison Modeste', 'Villa Premium', 'Studio Basique', 'Manoir de Luxe', 'Maison Moyenne']

# Prédiction
predictions = loaded_model.run(new_houses, function_name='predict')

print('\n🔮 Prédictions de prix pour 5 nouvelles maisons :')
print('─' * 55)
for label, pred in zip(house_labels, predictions):
    pred_val = pred if isinstance(pred, (int, float)) else float(pred[0])
    print(f'   {label:<20} → {pred_val:>12,.0f} ₹')
print('─' * 55)

In [ ]:
# -------------------------------------------------------------------
# 9.3 Sauvegarde des prédictions dans une table Snowflake
# -------------------------------------------------------------------
df_new_houses_pred = new_houses.copy()
df_new_houses_pred['LABEL'] = house_labels
df_new_houses_pred['PREDICTED_PRICE'] = predictions
df_new_houses_pred['PREDICTION_DATE'] = pd.Timestamp.now()
df_new_houses_pred.columns = [c.upper() for c in df_new_houses_pred.columns]

snow_preds = session.create_dataframe(df_new_houses_pred)
snow_preds.write.save_as_table(
    'HOUSE_PRICE_DB.ML.PREDICTIONS',
    mode='overwrite'
)

print('✅ Prédictions sauvegardées dans HOUSE_PRICE_DB.ML.PREDICTIONS')
session.table('HOUSE_PRICE_DB.ML.PREDICTIONS').show()

---
## ✅ Récapitulatif du Pipeline

| Étape | Statut | Description |
|-------|--------|-------------|
| 1. Config | ✅ | Session Snowpark initialisée |
| 2. Ingestion | ✅ | Données chargées depuis Snowflake |
| 3. EDA | ✅ | Distributions, corrélations analysées |
| 4. Features | ✅ | Encodage, normalisation, train/test split |
| 5. Entraînement | ✅ | 3 modèles entraînés |
| 6. Évaluation | ✅ | Métriques calculées et comparées |
| 7. Optimisation | ✅ | RandomizedSearchCV appliqué |
| 8. Registry | ✅ | Modèle sauvegardé dans le Snowflake Registry |
| 9. Inférence | ✅ | Prédictions générées et sauvegardées |

**Prochaine étape :** Déployer l'application Streamlit pour les utilisateurs métier (voir `streamlit/app.py`)